# Feature Engineering Pipeline

## Supply Chain Late Delivery Prediction

---

### Overview

Transform preprocessed data into ML-ready features while preventing data leakage.

### Data Leakage Prevention

**Critical**: These columns contain POST-DELIVERY information and MUST be excluded:

| Column | Reason | What It Contains |
|--------|--------|------------------|
| `late_delivery_risk` | Target variable | Whether delivery was late (1) or not (0) |
| `delivery_status` | Target in different form | Late delivery, Advance shipping, On time, etc. |
| `days_for_shipping_(real)` | Post-delivery info | Actual number of shipping days |
| `shipping_date` | Post-delivery info | When order actually shipped |

### Feature Engineering Strategy

| Category | Description | Example Features |
|----------|-------------|------------------|
| **Temporal** | Order timing patterns | day_of_week, month, quarter, is_weekend |
| **Customer** | Customer behavior metrics | order_count, lifetime_value, segment |
| **Product** | Product characteristics | popularity, category, price tier |
| **Shipping** | Shipping configuration | mode, scheduled_days, urgency_score |
| **Geographic** | Location-based features | market, region, country encoding |
| **Financial** | Order economics | profit_margin, discount_rate, order_value |

---

In [1]:
# ============================================================
# SETUP
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder

import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load preprocessed data
from src.data.preprocess import load_or_preprocess

df = load_or_preprocess()
print(f"Input data: {df.shape[0]:,} rows x {df.shape[1]} columns")

📂 Loading latest file: /Users/unclesam/Projects/supply-chain-ml-project/data/interim/cleaned_data_20251205_1114.parquet
✅ Loaded cached preprocessed data from data/interim
Input data: 180,519 rows x 56 columns


---

## 1. Define Leaky Columns

**Talking Point**: "Before engineering features, we identify columns that leak target information. Using these would give unrealistically high accuracy but fail in production."

In [3]:
# ============================================================
# LEAKAGE PREVENTION
# ============================================================

# Columns that MUST be excluded (contain post-delivery information)
LEAKY_COLUMNS = {
    'late_delivery_risk',           # Target variable itself
    'delivery_status',              # Categorical target
    'delivery_status_encoded',      # Encoded target
    'days_for_shipping_(real)',     # Only known after delivery
    'days_for_shipping_real',       # Alternative naming
    'delivery_days',                # Calculated from actual delivery
    'shipping_date_(dateorders)',   # Actual shipping date
    'shipping_date_dateorders',     # Alternative naming
}

# Find target column
target_col = None
for col in ['late_delivery_risk', 'late_delivery']:
    if col in df.columns:
        target_col = col
        break

print("LEAKAGE PREVENTION")
print("=" * 60)
print(f"Target column: {target_col}")
print(f"\nColumns excluded from features:")
excluded_in_data = [c for c in df.columns if c.lower() in [l.lower() for l in LEAKY_COLUMNS]]
for col in excluded_in_data:
    print(f"   - {col}")

LEAKAGE PREVENTION
Target column: late_delivery_risk

Columns excluded from features:
   - days_for_shipping_(real)
   - delivery_status
   - late_delivery_risk
   - shipping_date_(dateorders)


---

## 2. Temporal Features

**Talking Point**: "We extract time-based patterns from order dates - but NOT shipping dates, which would leak information."

In [ ]:
# ============================================================
# TEMPORAL FEATURES (from ORDER date only)
# ============================================================

# Find order date column
date_col = None
for col in ['order_date_(dateorders)', 'order_date_dateorders', 'order_date']:
    if col in df.columns:
        date_col = col
        break

temporal_features = []

if date_col:
    # Ensure datetime
    if not pd.api.types.is_datetime64_any_dtype(df[date_col]):
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # Extract temporal features
    df['order_day_of_week'] = df[date_col].dt.dayofweek
    df['order_month'] = df[date_col].dt.month
    df['order_quarter'] = df[date_col].dt.quarter
    df['is_weekend'] = (df['order_day_of_week'] >= 5).astype(int)
    df['order_hour'] = df[date_col].dt.hour

    temporal_features = ['order_day_of_week', 'order_month', 'order_quarter', 'is_weekend', 'order_hour']

    print(f"Temporal features created from {date_col}:")
    for feat in temporal_features:
        print(f"   - {feat}: {df[feat].nunique()} unique values")
else:
    print("Warning: Order date column not found")

Temporal features created from order_date_(dateorders):
   - order_day_of_week: 7 unique values
   - order_month: 12 unique values
   - order_quarter: 4 unique values
   - is_weekend: 2 unique values
   - order_hour: 24 unique values


---

## 3. Customer Features

**Talking Point**: "Customer behavior metrics help identify patterns - frequent buyers may have different delivery patterns."

In [ ]:
# ============================================================
# CUSTOMER FEATURES
# ============================================================

customer_features = []

# Find customer ID column
customer_id_col = None
for col in ['customer_id', 'customerid', 'order_customer_id']:
    if col in df.columns:
        customer_id_col = col
        break

if customer_id_col:
    # Customer order frequency
    customer_counts = df.groupby(customer_id_col).size()
    df['customer_order_count'] = df[customer_id_col].map(customer_counts)
    customer_features.append('customer_order_count')

    # Customer lifetime value (total sales)
    sales_col = [c for c in df.columns if c.lower() == 'sales']
    if sales_col:
        customer_sales = df.groupby(customer_id_col)[sales_col[0]].sum()
        df['customer_lifetime_value'] = df[customer_id_col].map(customer_sales)
        customer_features.append('customer_lifetime_value')

    print(f"Customer features created:")
    for feat in customer_features:
        print(f"   - {feat}: min={df[feat].min():.2f}, max={df[feat].max():.2f}, mean={df[feat].mean():.2f}")
else:
    print("Warning: Customer ID column not found")

Customer features created:
   - customer_order_count: min=1.00, max=47.00, mean=16.77
   - customer_lifetime_value: min=11.29, max=10524.17, mean=3341.21


---

## 4. Product Features

**Talking Point**: "Product characteristics like popularity and category help capture item-specific delivery patterns."

In [6]:
# ============================================================
# PRODUCT FEATURES
# ============================================================

product_features = []

# Product popularity
product_col = [c for c in df.columns if 'product_name' in c.lower()]
if product_col:
    product_counts = df.groupby(product_col[0]).size()
    df['product_popularity'] = df[product_col[0]].map(product_counts)
    product_features.append('product_popularity')

# Category popularity
category_col = [c for c in df.columns if 'category_name' in c.lower()]
if category_col:
    category_counts = df.groupby(category_col[0]).size()
    df['category_popularity'] = df[category_col[0]].map(category_counts)
    product_features.append('category_popularity')

# Order value
price_col = [c for c in df.columns if 'product_price' in c.lower()]
qty_col = [c for c in df.columns if 'order_item_quantity' in c.lower()]
if price_col and qty_col:
    df['order_value'] = df[price_col[0]] * df[qty_col[0]]
    product_features.append('order_value')

# Discount rate
discount_col = [c for c in df.columns if 'discount' in c.lower() and 'rate' in c.lower()]
if discount_col:
    df['discount_rate'] = df[discount_col[0]].clip(0, 1)
    product_features.append('discount_rate')

print(f"Product features created:")
for feat in product_features:
    print(f"   - {feat}: mean={df[feat].mean():.2f}")

Product features created:
   - product_popularity: mean=16113.54
   - category_popularity: mean=16311.78
   - order_value: mean=203.77
   - discount_rate: mean=0.10


---

## 5. Shipping Features

**Talking Point**: "Shipping configuration is critical - but we only use SCHEDULED days, not actual delivery time."

In [7]:
# ============================================================
# SHIPPING FEATURES (Pre-delivery only!)
# ============================================================

shipping_features = []

# Shipping mode urgency score
shipping_mode_col = [c for c in df.columns if 'shipping_mode' in c.lower()]
if shipping_mode_col:
    urgency_map = {
        'Same Day': 4,
        'First Class': 3,
        'Second Class': 2,
        'Standard Class': 1
    }
    df['shipping_urgency'] = df[shipping_mode_col[0]].map(urgency_map).fillna(1)
    shipping_features.append('shipping_urgency')

# Scheduled shipping days (SAFE - this is the promised time, not actual)
scheduled_col = [c for c in df.columns if 'scheduled' in c.lower() and 'day' in c.lower()]
if scheduled_col:
    df['scheduled_shipping_days'] = df[scheduled_col[0]]
    shipping_features.append('scheduled_shipping_days')

print(f"Shipping features created (pre-delivery info only):")
for feat in shipping_features:
    print(f"   - {feat}: mean={df[feat].mean():.2f}")

print(f"\nNOTE: 'days_for_shipping_(real)' is EXCLUDED - only known after delivery!")

Shipping features created (pre-delivery info only):
   - shipping_urgency: mean=1.67
   - scheduled_shipping_days: mean=2.93

NOTE: 'days_for_shipping_(real)' is EXCLUDED - only known after delivery!


---

## 6. Financial Features

**Talking Point**: "Financial metrics help identify high-value orders that may warrant priority handling."

In [8]:
# ============================================================
# FINANCIAL FEATURES
# ============================================================

financial_features = []

# Profit margin percentage
profit_col = [c for c in df.columns if 'profit' in c.lower() and 'order' in c.lower()]
sales_col = [c for c in df.columns if c.lower() == 'sales']

if profit_col and sales_col:
    df['profit_margin_pct'] = (df[profit_col[0]] / (df[sales_col[0]] + 1e-6)) * 100
    df['profit_margin_pct'] = df['profit_margin_pct'].clip(-100, 100)
    financial_features.append('profit_margin_pct')

# Sales per item
qty_col = [c for c in df.columns if 'order_item_quantity' in c.lower()]
if sales_col and qty_col:
    df['sales_per_item'] = df[sales_col[0]] / (df[qty_col[0]] + 1e-6)
    financial_features.append('sales_per_item')

# High value order flag
if sales_col:
    threshold = df[sales_col[0]].quantile(0.75)
    df['is_high_value'] = (df[sales_col[0]] > threshold).astype(int)
    financial_features.append('is_high_value')

print(f"Financial features created:")
for feat in financial_features:
    print(f"   - {feat}: mean={df[feat].mean():.2f}")

Financial features created:
   - profit_margin_pct: mean=0.10
   - sales_per_item: mean=139.51
   - is_high_value: mean=0.23


---

## 7. Categorical Encoding

**Talking Point**: "We encode categorical variables numerically. Critically, we EXCLUDE delivery_status as it's the target."

In [ ]:
# ============================================================
# CATEGORICAL ENCODING
# ============================================================

# Categorical columns to encode (EXCLUDING leaky columns)
categorical_cols = [
    'type', 'category_name', 'customer_segment', 'department_name',
    'market', 'order_region', 'order_country', 'order_state',
    'order_city', 'shipping_mode'
]

# Filter to columns that exist
categorical_cols = [c for c in categorical_cols if c in df.columns]

# IMPORTANT: Exclude delivery_status - it's the target!
categorical_cols = [c for c in categorical_cols if 'delivery_status' not in c.lower()]

encoded_features = []
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    encoded_col = f"{col}_encoded"
    df[encoded_col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    encoded_features.append(encoded_col)

print(f"Encoded {len(encoded_features)} categorical features:")
for feat in encoded_features:
    orig = feat.replace('_encoded', '')
    print(f"   - {feat}: {df[orig].nunique()} categories")

print(f"\nNOTE: 'delivery_status' is EXCLUDED - it leaks the target!")

Encoded 10 categorical features:
   - type_encoded: 4 categories
   - category_name_encoded: 50 categories
   - customer_segment_encoded: 3 categories
   - department_name_encoded: 11 categories
   - market_encoded: 5 categories
   - order_region_encoded: 23 categories
   - order_country_encoded: 164 categories
   - order_state_encoded: 1089 categories
   - order_city_encoded: 3597 categories
   - shipping_mode_encoded: 4 categories

NOTE: 'delivery_status' is EXCLUDED - it leaks the target!


---

## 8. Assemble Feature Matrix

**Talking Point**: "Let's combine all features into our final feature matrix, ensuring no leaky columns are included."

In [ ]:
# ============================================================
# ASSEMBLE FEATURE MATRIX
# ============================================================

# Combine all feature lists
all_features = (temporal_features + customer_features + product_features +
                shipping_features + financial_features + encoded_features)

# Filter to features that exist in dataframe
all_features = [f for f in all_features if f in df.columns]

# FINAL LEAKAGE CHECK
safe_features = []
for feat in all_features:
    is_leaky = any(leaky.lower() in feat.lower() for leaky in ['delivery_status', 'days_for_shipping_real', 'late_delivery'])
    if not is_leaky:
        safe_features.append(feat)
    else:
        print(f"BLOCKED leaky feature: {feat}")

# Create feature matrix
X = df[safe_features].copy()
X = X.fillna(0)  # Handle any remaining NaN

# Create target
y = df[target_col].astype(int)

print(f"\nFEATURE MATRIX CREATED")
print("=" * 60)
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]:,}")
print(f"\nFeature list:")
for i, feat in enumerate(X.columns, 1):
    print(f"   {i:2d}. {feat}")


FEATURE MATRIX CREATED
Features: 26
Samples: 180,519

Feature list:
    1. order_day_of_week
    2. order_month
    3. order_quarter
    4. is_weekend
    5. order_hour
    6. customer_order_count
    7. customer_lifetime_value
    8. product_popularity
    9. category_popularity
   10. order_value
   11. discount_rate
   12. shipping_urgency
   13. scheduled_shipping_days
   14. profit_margin_pct
   15. sales_per_item
   16. is_high_value
   17. type_encoded
   18. category_name_encoded
   19. customer_segment_encoded
   20. department_name_encoded
   21. market_encoded
   22. order_region_encoded
   23. order_country_encoded
   24. order_state_encoded
   25. order_city_encoded
   26. shipping_mode_encoded


---

## 9. Feature Correlation Analysis

**Talking Point**: "Let's visualize which features are most predictive of late delivery."

In [11]:
# ============================================================
# FEATURE CORRELATION ANALYSIS
# ============================================================

# Calculate correlations with target
correlations = X.corrwith(y).sort_values(key=abs, ascending=False)

# Create correlation bar chart
fig = go.Figure()

colors = ['#e74c3c' if v > 0 else '#3498db' for v in correlations.values]

fig.add_trace(go.Bar(
    y=correlations.index,
    x=correlations.values,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in correlations.values],
    textposition='outside'
))

fig.update_layout(
    title='<b>Feature Correlation with Late Delivery</b>',
    xaxis_title='Correlation Coefficient',
    yaxis={'categoryorder': 'total ascending'},
    height=max(500, len(correlations) * 25),
    showlegend=False
)

fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()

# Dynamic interpretation
print("\nTOP PREDICTORS OF LATE DELIVERY:")
print("=" * 60)
for i, (feat, corr) in enumerate(correlations.head(5).items(), 1):
    direction = "increases" if corr > 0 else "decreases"
    print(f"{i}. {feat}: {corr:.3f}")
    print(f"   Higher values {direction} late delivery risk")


TOP PREDICTORS OF LATE DELIVERY:
1. shipping_mode_encoded: -0.401
   Higher values decreases late delivery risk
2. scheduled_shipping_days: -0.369
   Higher values decreases late delivery risk
3. shipping_urgency: 0.332
   Higher values increases late delivery risk
4. type_encoded: -0.062
   Higher values decreases late delivery risk
5. order_hour: 0.047
   Higher values increases late delivery risk


In [12]:
# Feature correlation heatmap (top features only)
top_features = correlations.head(10).index.tolist()
top_features.append(target_col)

corr_matrix = df[top_features].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}',
    textfont={'size': 10}
))

fig.update_layout(
    title='<b>Correlation Heatmap: Top Features vs Target</b>',
    height=600,
    width=700,
    xaxis={'tickangle': 45},
    yaxis={'autorange': 'reversed'}
)
fig.show()

---

## 10. Save Features & Summary

**Talking Point**: "Our feature engineering is complete. We have 26+ leakage-free features ready for modeling."

In [13]:
# ============================================================
# SAVE FEATURES
# ============================================================
from pathlib import Path
from datetime import datetime

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Save feature matrix
X.to_parquet(output_dir / f'features_{timestamp}.parquet', index=False)
y.to_frame().to_parquet(output_dir / f'target_{timestamp}.parquet', index=False)

print(f"Features saved to: {output_dir}")

Features saved to: ../data/processed


In [14]:
# Final summary
print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)

late_rate = y.mean() * 100

print(f"""
SUMMARY
{'='*60}

OUTPUT:
   Features: {X.shape[1]} (all leakage-free)
   Samples: {X.shape[0]:,}
   Target: {late_rate:.1f}% late delivery rate

FEATURE CATEGORIES:
   Temporal: {len(temporal_features)} features
   Customer: {len(customer_features)} features
   Product: {len(product_features)} features
   Shipping: {len(shipping_features)} features
   Financial: {len(financial_features)} features
   Encoded: {len(encoded_features)} features

TOP PREDICTORS:
   1. {correlations.index[0]}: {correlations.iloc[0]:.3f}
   2. {correlations.index[1]}: {correlations.iloc[1]:.3f}
   3. {correlations.index[2]}: {correlations.iloc[2]:.3f}

LEAKAGE PREVENTION:
   Excluded: delivery_status, days_for_shipping_(real)
   All features are available at ORDER TIME

{'='*60}
Next: Run 04_model_training.ipynb
""")


FEATURE ENGINEERING COMPLETE

SUMMARY

OUTPUT:
   Features: 26 (all leakage-free)
   Samples: 180,519
   Target: 54.8% late delivery rate

FEATURE CATEGORIES:
   Temporal: 5 features
   Customer: 2 features
   Product: 4 features
   Shipping: 2 features
   Financial: 3 features
   Encoded: 10 features

TOP PREDICTORS:
   1. shipping_mode_encoded: -0.401
   2. scheduled_shipping_days: -0.369
   3. shipping_urgency: 0.332

LEAKAGE PREVENTION:
   Excluded: delivery_status, days_for_shipping_(real)
   All features are available at ORDER TIME

Next: Run 04_model_training.ipynb

